# 13551224 형상 비교 시험
기존 모델 재학습 없이 학습 심볼 한 장을 PDF 첫 페이지 전체에서 찾습니다. 모든 셀을 실행하세요.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, math
import cv2
import numpy as np
import pymupdf as fitz
from PIL import Image
from IPython.display import display

DATASET = Path(r"C:\Users\Win11Pro\Desktop\drawing\yolo_dataset_random_80_20")
PDF_PATH = Path(r"C:\Users\Win11Pro\Desktop\1. DWG 객체 변환 도면 생성 샘플\DWG 객체 변환 샘플.pdf")
IMAGE = DATASET / "images/train/train_000027.png"
LABEL = DATASET / "labels/train/train_000027.txt"
OUT = Path.cwd() / "template_match_trial_output" / datetime.now().strftime("%Y%m%d_%H%M%S")
DPI, TILE, STEP = 144, 960, 480
SCALES = (0.90, 0.94, 0.98)
MIN_SCORE = 0.25

for path in (PDF_PATH, IMAGE, LABEL):
    if not path.is_file():
        raise FileNotFoundError(path)
OUT.mkdir(parents=True, exist_ok=False)

im = cv2.imread(str(IMAGE), cv2.IMREAD_GRAYSCALE)
class_id, cx, cy, bw, bh = map(float, LABEL.read_text().split())
assert int(class_id) == 11, "클래스 11 = 13551224 기준 이미지가 아닙니다."
h, w = im.shape
x1, y1, x2, y2 = (round(v) for v in (
    (cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h
))
template = im[y1:y2, x1:x2]
assert template.size
print("기준 심볼:", template.shape[::-1], "결과 폴더:", OUT)
display(Image.fromarray(template))
references = []
for scale in SCALES:
    resized = cv2.resize(template, None, fx=scale, fy=scale,
                         interpolation=cv2.INTER_AREA)
    references.append((scale, cv2.Canny(resized, 70, 180)))

def starts(length):
    if length <= TILE:
        return [0]
    values = list(range(0, length-TILE+1, STEP))
    if values[-1] != length-TILE:
        values.append(length-TILE)
    return values

hits = []
with fitz.open(PDF_PATH) as doc:
    page = doc[0]
    assert page.rotation == 0, "회전하지 않은 PDF 첫 페이지용입니다."
    drawing = page.get_displaylist(annots=False)
    factor = DPI / 72
    xs = starts(math.ceil(page.rect.width*factor))
    ys = starts(math.ceil(page.rect.height*factor))
    total = len(xs)*len(ys)
    checked = 0
    print("검사할 영역:", total)
    for top in ys:
        for left in xs:
            clip = fitz.Rect(left/factor, top/factor,
                min((left+TILE)/factor, page.rect.width),
                min((top+TILE)/factor, page.rect.height))
            pix = drawing.get_pixmap(matrix=fitz.Matrix(factor, factor),
                clip=clip, colorspace=fitz.csGRAY, alpha=False)
            checked += 1
            gray = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width)
            if gray.min() > 245:
                continue
            edges = cv2.Canny(gray, 70, 180)
            # shortcut: 타일마다 최고점 한 개만 찾음; 같은 타일의 다중 인스턴스는 별도 확장이 필요함.
            for scale, ref in references:
                if ref.shape[0] > edges.shape[0] or ref.shape[1] > edges.shape[1]:
                    continue
                response = cv2.matchTemplate(edges, ref, cv2.TM_CCOEFF_NORMED)
                _, score, _, (px, py) = cv2.minMaxLoc(response)
                if score >= MIN_SCORE:
                    x = (pix.x+px)/factor
                    y = (pix.y+py)/factor
                    hits.append({"score": round(float(score), 4),
                        "scale": scale, "bbox_pdf_pt": [x, y,
                        x+ref.shape[1]/factor, y+ref.shape[0]/factor]})
        print(f"완료 {checked}/{total}, 후보 {len(hits)}개")

    hits.sort(key=lambda item: item["score"], reverse=True)
    matches = []
    for hit in hits:
        x, y = hit["bbox_pdf_pt"][:2]
        if any(abs(x-m["bbox_pdf_pt"][0]) < 40 and
               abs(y-m["bbox_pdf_pt"][1]) < 150 for m in matches):
            continue
        matches.append(hit)
    for hit in matches:
        rect = fitz.Rect(hit["bbox_pdf_pt"])
        page.draw_rect(rect, color=(1, 0, 0), width=1.5)
        page.insert_text((rect.x0, max(6, rect.y0-3)),
            f"13551224 score={hit['score']:.3f}", fontsize=7, color=(1, 0, 0))
    pdf_out = OUT / "template_matches.pdf"
    doc.save(pdf_out, garbage=4, deflate=True)
    if matches:
        rect = fitz.Rect(matches[0]["bbox_pdf_pt"])
        crop = fitz.Rect(rect.x0-30, rect.y0-30,
                         rect.x1+30, rect.y1+30) & page.rect
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), clip=crop, alpha=False)
        pix.save(OUT / "best_match.png")

(OUT / "matches.json").write_text(json.dumps({
    "source_pdf": str(PDF_PATH), "class_name": "13551224",
    "min_score": MIN_SCORE, "matches": matches
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("중복 제거 후:", len(matches), "개")
for hit in matches[:10]:
    print(hit)
print("표시된 PDF:", pdf_out)
if matches:
    display(Image.open(OUT / "best_match.png"))
